<!-- SPDX-License-Identifier: GPL-3.0-or-later -->
# Dynamic spectrum of the hackathon recordings

Successor of the hackathon's `get_dynamic_spectrum.ipynb`. That notebook ported `mtcsd`
(labbox, Mitra & Harris) and AR whitening, but ran them on a course dataset
(`ws_data_1shank.mat`, hippocampal LFP) that was never in the repo. The code now lives,
tested and with two bugs fixed, in `ntxbci.spectral`; this notebook applies it to **our**
Ganglion recordings.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from ntxbci.events import detect_blinks
from ntxbci.io import load_recording
from ntxbci.manifest import load_manifest
from ntxbci.processing import clean
from ntxbci.spectral import dynamic_spectrum, whiten

manifest = load_manifest()
[e.id for e in manifest.entries]

## Blink test: the only recording with known events

In [ ]:
blink_entry = manifest.get("blink-test")
rec = load_recording(manifest.file(blink_entry))
blinks = [b.time_s for b in detect_blinks(rec)]
print(f"{rec.duration:.1f}s, detected blinks: {np.round(blinks, 2)}")
print("protocol:", blink_entry.events.times_s)

In [ ]:
x = clean(rec).mean(axis=1)
ds = dynamic_spectrum(x, rec.fs, nperseg=256).band(1, 40)
fig, ax = plt.subplots(figsize=(10, 3))
ax.pcolormesh(
    ds.times_s, ds.freqs_hz, 10 * np.log10(ds.power.T + 1e-12), shading="auto", cmap="magma"
)
for b in blinks:
    ax.axvline(b, color="w", lw=0.6, ls=":")
ax.set(
    xlabel="time [s]",
    ylabel="Hz",
    title="blink-test: blinks show up as broadband low-frequency bursts",
)
plt.show()

## AR(2) pre-whitening on a session

Whitening flattens the 1/f background so that band-specific changes (e.g. alpha) stand out,
which is what the original notebook did before computing the spectrum.

In [ ]:
session = load_recording(manifest.file(manifest.get("session4")))
raw = clean(session)[:, 0]
white = whiten(raw, order=2)
ds_raw = dynamic_spectrum(raw, session.fs, nperseg=256).band(1, 40)
ds_white = dynamic_spectrum(white, session.fs, nperseg=256).band(1, 40)
fig, axes = plt.subplots(1, 2, figsize=(10, 3), sharey=True)
for ax, d, title in zip(axes, (ds_raw, ds_white), ("cleaned", "AR(2)-whitened"), strict=True):
    ax.semilogy(d.freqs_hz, d.power.mean(axis=0))
    ax.set(title=f"session4 ch1, {title}", xlabel="Hz")
axes[0].set_ylabel("power")
plt.show()

In [ ]:
alpha = ds_white.band(8, 13).power.mean(axis=1)
print(f"alpha power over time: min={alpha.min():.3g} max={alpha.max():.3g} segments={len(alpha)}")
assert len(alpha) == len(ds_white.times_s)